# Researcher's Guide: Advanced Optimization Workflows

This notebook covers advanced hyperparameter optimization patterns for research workflows.
It builds on the fundamentals covered in [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb).

**Prerequisites**: Familiarity with `run_hyperparameter_optimization()` and `run_optimize_and_evaluate()` from the hyperparameter tuning notebook.

**What this notebook covers**:
1. **Discovering** compatible planners and available metrics for any environment
2. **Multi-objective optimization** — optimizing for multiple metrics simultaneously
3. **Risk-averse optimization** — using safety-focused metrics per environment
4. **Interpreting results** — understanding the statistics DataFrame columns
5. **Workflow tips** — categorical parameters, iterative refinement, caching

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from POMDPPlanners.simulations.simulation_apis.local_simulations_api import LocalSimulationsAPI
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs
from POMDPPlanners.core.simulation import NumericalHyperParameter, CategoricalHyperParameter
from POMDPPlanners.core.simulation.hyperparameter_tuning import (
    HyperParamPlannerConfig,
    HyperParameterRunParams,
    HyperParameterOptimizationDirection,
)
from POMDPPlanners.simulations.simulation_statistics import get_available_optimization_metrics
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP

np.random.seed(42)

# Shared setup
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
planner_configs = PlannersHyperparamConfigs(discount_factor=0.95)
api = LocalSimulationsAPI(debug=True)

# Create environment and belief
tiger_env, tiger_belief = env_configs.tiger_pomdp_config(n_particles=50)

print(f"Environment: {tiger_env.__class__.__name__}")
print(f"Belief particles: {len(tiger_belief.particles)}")

/home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/kobi/Documents/github/POMDPPlanners/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-25 13:58:39,697 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulation_apis/local_simulations_api.py:103 - Initialized LocalSimulationsAPI
2026-02-25 13:58:39,697 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing TigerPOMDP environment with discount factor 0.95


Environment: TigerPOMDP
Belief particles: 50


## 1. Discovering Compatible Planners and Metrics

Before setting up optimization, use the config APIs to discover what planners are compatible with your environment and what metrics you can optimize.

In [2]:
# Discover all compatible planners for the environment
compatible_planners = planner_configs.get_compatible_planners(tiger_env)

print(f"Compatible planners for {tiger_env.__class__.__name__}:\n")
for config in compatible_planners:
    hp_names = [hp.name for hp in config.hyper_parameters]
    print(f"  {config.policy_cls.__name__}")
    print(f"    Tunable: {hp_names}")
    print(f"    Fixed:   {list(config.constant_parameters.keys())}\n")

Compatible planners for TigerPOMDP:

  POMCPOW
    Tunable: ['exploration_constant', 'depth', 'k_a', 'alpha_a', 'k_o', 'alpha_o']
    Fixed:   ['discount_factor', 'name', 'environment', 'action_sampler', 'time_out_in_seconds']

  POMCP_DPW
    Tunable: ['exploration_constant', 'depth', 'k_a', 'alpha_a', 'k_o', 'alpha_o']
    Fixed:   ['discount_factor', 'name', 'environment', 'action_sampler', 'time_out_in_seconds']

  POMCP
    Tunable: ['exploration_constant', 'depth']
    Fixed:   ['discount_factor', 'name', 'environment', 'time_out_in_seconds']

  SparsePFT
    Tunable: ['depth', 'c_ucb', 'beta_ucb', 'belief_child_num']
    Fixed:   ['discount_factor', 'gamma', 'name', 'environment', 'time_out_in_seconds']

  SparseSamplingDiscreteActionsPlanner
    Tunable: ['branching_factor', 'depth']
    Fixed:   ['environment', 'name']

  DiscreteActionSequencesPlanner
    Tunable: ['depth', 'n_return_samples']
    Fixed:   ['discount_factor', 'name', 'environment']



In [3]:
# Discover available optimization metrics for a specific planner
metrics = get_available_optimization_metrics(tiger_env, POMCP)

print(f"Available optimization metrics for {tiger_env.__class__.__name__} + POMCP:\n")
for metric in metrics:
    print(f"  - {metric}")

Available optimization metrics for TigerPOMDP + POMCP:

  - average_return
  - return_cvar
  - return_value_at_risk
  - average_state_sampling_time
  - average_action_time
  - average_observation_time
  - average_belief_update_time
  - average_reward_time
  - average_actual_num_steps
  - average_reach_terminal_state
  - success_rate
  - average_listens
  - policy_info_min_actions_visit_count
  - policy_info_max_actions_visit_count
  - policy_info_actions_visit_count_entropy
  - policy_info_n_actions_from_root
  - policy_info_root_visit_count
  - policy_info_tree_max_depth
  - policy_info_is_leaf


## 2. Multi-Objective Optimization

By default, `hyperparameter_tuning.ipynb` optimizes a single metric (`average_return`). In research, you often want to optimize multiple objectives simultaneously — for example, maximizing return *and* success rate.

Pass multiple entries to `parameters_to_optimize` to enable multi-objective optimization via Optuna:

| Environment | Standard Metrics |
|---|---|
| TigerPOMDP | `average_return`, `success_rate` |
| CartPolePOMDP | `average_return`, `goal_reaching_rate` |
| MountainCarPOMDP | `average_return`, `goal_reaching_rate` |
| LightDarkPOMDP (discrete/continuous) | `average_return`, `goal_reaching_rate` |
| PushPOMDP | `average_return`, `goal_reaching_rate` |
| RockSamplePOMDP | `average_return`, `exit_success_rate` |
| LaserTagPOMDP (discrete/continuous) | `average_return`, `tag_success_rate` |
| PacManPOMDP | `average_return`, `win_rate` |

In [ ]:
# Multi-objective: optimize for both average_return and success_rate
multi_obj_config = [
    HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        hyper_param_planner_config=planner_configs.pomcp_config(
            env=tiger_env, name="MultiObj_POMCP"
        ),
        num_episodes=10,
        num_steps=20,
        n_trials=20,
        parameters_to_optimize=[
            ("average_return", HyperParameterOptimizationDirection.MAXIMIZE),
            ("success_rate", HyperParameterOptimizationDirection.MAXIMIZE),
        ],
    )
]

results, stats_df = api.run_optimize_and_evaluate(
    configs=multi_obj_config,
    evaluation_episodes=30,
    evaluation_steps=20,
    evaluation_n_jobs=-1,
    optimization_n_jobs=-1,
    cache_dir_path=Path("./researcher_cache"),
    experiment_name="Multi_Objective_Tiger",
)

print("Multi-Objective Optimization Results:")
print(stats_df.to_string())

2026-02-25 14:01:59,236 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulation_apis/local_simulations_api.py:819 - Starting optimize and evaluate workflow with local execution
2026-02-25 14:01:59,236 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/workflows/hyperparameter_tuning_evaluation_workflows.py:166 - Optimizing 1 environments and policies
2026/02/25 14:01:59 INFO mlflow.tracking.fluent: Experiment with name 'Multi_Objective_Tiger' does not exist. Creating a new experiment.
2026-02-25 14:01:59,278 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:523 - Logging to file: researcher_cache/task_manager_cache/logs/disk_cache_db_20260225_140159.log
2026-02-25 14:01:59,279 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/hyper_parameter_tuning_simulations.py:424 - Starting optimization for 1 configurations using stub interface with MLflow tracking
2026-02-25 14:01:59,375

## 3. Risk-Averse Optimization

Standard optimization maximizes expected return. Risk-averse optimization targets safety-related metrics instead — minimizing obstacle collisions, dangerous encounters, or safety violations.

Each environment defines its own risk metrics:

| Environment | Risk Metrics | Direction |
|---|---|---|
| TigerPOMDP | `success_rate` | MAXIMIZE |
| ContinuousLightDarkPOMDP | `avg_obstacle_hit_counter` | MINIMIZE |
| | `goal_reaching_rate` | MAXIMIZE |
| DiscreteLightDarkPOMDP | `avg_obstacle_hit_counter` | MINIMIZE |
| | `goal_reaching_rate` | MAXIMIZE |
| PushPOMDP | `total_all_obstacle_collisions` | MINIMIZE |
| | `goal_reaching_rate` | MAXIMIZE |
| RockSamplePOMDP | `average_dangerous_area_steps` | MINIMIZE |
| | `exit_success_rate` | MAXIMIZE |
| LaserTagPOMDP | `average_all_dangerous_encounters` | MINIMIZE |
| | `tag_success_rate` | MAXIMIZE |
| PacManPOMDP | `avg_collision_encounters` | MINIMIZE |
| | `win_rate` | MAXIMIZE |
| SafeAntVelocityPOMDP | `total_safety_violations` | MINIMIZE |

To use risk-averse optimization, set `parameters_to_optimize` to the risk metrics for your environment instead of `average_return`:

In [5]:
# Risk-averse: optimize success_rate only (ignoring return)
risk_averse_config = [
    HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        hyper_param_planner_config=planner_configs.pomcp_config(
            env=tiger_env, name="RiskAverse_POMCP"
        ),
        num_episodes=10,
        num_steps=20,
        n_trials=20,
        parameters_to_optimize=[
            ("success_rate", HyperParameterOptimizationDirection.MAXIMIZE),
        ],
    )
]

risk_results, risk_stats_df = api.run_optimize_and_evaluate(
    configs=risk_averse_config,
    evaluation_episodes=30,
    evaluation_steps=20,
    evaluation_n_jobs=-1,
    optimization_n_jobs=-1,
    cache_dir_path=Path("./researcher_cache"),
    experiment_name="Risk_Averse_Tiger",
)

print("Risk-Averse Optimization Results:")
print(risk_stats_df.to_string())

## 4. Interpreting the Results DataFrame

`run_optimize_and_evaluate()` returns a `(results_dict, stats_df)` tuple. The DataFrame contains one row per environment-policy pair with the following column structure:

**Identification columns:**
- `environment` — environment name
- `policy` — policy name

**For each metric `X`, three columns:**
- `X` — point estimate
- `X_ci_lower` — lower confidence interval bound
- `X_ci_upper` — upper confidence interval bound

**Standard metrics** (always present):

| Metric | Description |
|---|---|
| `average_return` | Mean discounted cumulative reward |
| `return_cvar` | Conditional Value at Risk (expected return in the worst percentile of episodes) |
| `return_value_at_risk` | Value at Risk (return quantile) |
| `average_action_time` | Mean wall-clock time per planning step |
| `average_belief_update_time` | Mean time to update belief after observation |
| `average_actual_num_steps` | Mean episode length |
| `average_reach_terminal_state` | Fraction of episodes reaching terminal state |

**Environment-specific metrics** vary by environment (e.g., `success_rate`, `goal_reaching_rate`, `avg_obstacle_hit_counter`).

**Policy-specific metrics** are prefixed with `policy_info_` (e.g., `policy_info_min_actions_visit_count` for POMCP).

In [6]:
# Combine results from both optimization runs for comparison
combined_df = pd.concat([stats_df, risk_stats_df], ignore_index=True)

# Show all available columns
print("Available columns:")
print(combined_df.columns.tolist())

# Compare key metrics between multi-objective and risk-averse policies
key_cols = [
    "policy", "average_return", "average_return_ci_lower", "average_return_ci_upper",
    "success_rate", "success_rate_ci_lower", "success_rate_ci_upper",
    "return_cvar", "average_action_time",
]
display_cols = [c for c in key_cols if c in combined_df.columns]

print("\nPolicy Comparison (multi-objective vs risk-averse):")
print(combined_df[display_cols].to_string(index=False))

## 5. Research Workflow Tips

### Categorical Hyperparameters

Use `CategoricalHyperParameter` when you want to test specific discrete values rather than searching a continuous range:

In [7]:
# Instead of searching a continuous range:
#   NumericalHyperParameter(2, 10, "depth")
# Test specific depth values:
cat_depth = CategoricalHyperParameter([3, 5, 7, 10], "depth")

# Mix numerical and categorical in the same config
mixed_config = HyperParamPlannerConfig(
    policy_cls=POMCP,
    hyper_parameters=[
        NumericalHyperParameter(0.1, 100.0, "exploration_constant"),
        CategoricalHyperParameter([3, 5, 7, 10], "depth"),
    ],
    constant_parameters={
        "discount_factor": 0.95,
        "n_simulations": 500,
        "name": "POMCP_CategoricalDepth",
    },
)

print(f"Hyperparameters: {[hp.name for hp in mixed_config.hyper_parameters]}")
print(f"  exploration_constant: continuous [{mixed_config.hyper_parameters[0].low}, {mixed_config.hyper_parameters[0].high}]")
print(f"  depth: categorical {mixed_config.hyper_parameters[1].choices}")

### Iterative Refinement Workflow

1. **Quick scan** — Run with small `n_trials` (10-20) and few episodes (5-10) to validate the setup and get rough parameter ranges
2. **Inspect results** — Check `chosen_hyper_parameters` from `OptimizedPolicyResult`. Are values hitting range boundaries? If so, widen the range
3. **Refine ranges** — Narrow `NumericalHyperParameter` ranges around promising regions
4. **Full run** — Increase `n_trials` (100-500) and `evaluation_episodes` (50-200) for statistically reliable results

### Caching

Results are cached based on configuration. Same configuration = reused results. To force a fresh run:
- Change `experiment_name`
- Use a different `cache_dir_path`

### Parallel Execution

- `optimization_n_jobs=-1` — use all CPU cores for optimization trials
- `evaluation_n_jobs=-1` — use all CPU cores for evaluation episodes
- Both can be set independently depending on available resources

## What's Next?

- **Basic usage**: See [basic_usage.ipynb](basic_usage.ipynb) for the core POMDP interaction loop and manual episode execution
- **Hyperparameter tuning fundamentals**: See [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for single-planner optimization and `run_optimize_and_evaluate()`
- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for batch evaluation of pre-configured policies with statistical analysis
- **Inspect planner behavior**: See [tree_analysis_debugging.ipynb](tree_analysis_debugging.ipynb) for MCTS tree metrics, visualization, and debugging
- **Belief representations**: See [belief_representations.ipynb](belief_representations.ipynb) for particle, Gaussian, and GMM beliefs